# LeetCode 3231 — Minimum Number of Increasing Subsequence to Be Removed bằng Python

- Link: https://leetcode.com/problems/minimum-number-of-increasing-subsequence-to-be-removed/
- Độ khó: **Hard** · **Premium**
- Chủ đề: **Định lý Dilworth / Mirsky**: số dãy con tăng nghiêm ngặt ít nhất phủ mảng = độ dài **dãy con không tăng** dài nhất; tham lam "xếp bài" với **tìm kiếm nhị phân** `O(n log n)`

> **Ghi chú Premium:** bài Premium nên LeetCode không trả đề qua API công khai. Đề, ví dụ, ràng buộc và chữ ký `minOperations(self, nums)` lấy theo trang doocs (https://leetcode.doocs.org/en/lc/3231/) — vui lòng **đối chiếu lại** nếu có tài khoản Premium.

Skeleton (theo doocs):

```python
class Solution:
    def minOperations(self, nums: List[int]) -> int:
```

## 1. Mục tiêu học

1. Mỗi thao tác xóa một dãy con **tăng nghiêm ngặt** ⇒ cần phân hoạch mảng thành ít dãy con tăng nghiêm ngặt nhất.
2. Hai phần tử `i < j` với `nums[i] ≥ nums[j]` **không thể** cùng một dãy ⇒ một dãy con **không tăng** dài `L` cần ít nhất `L` thao tác (cận dưới).
3. Tham lam đạt đúng cận dưới: duyệt trái sang phải, đặt `x` vào dãy có phần tử cuối **lớn nhất mà vẫn `< x`**; nếu không có, mở dãy mới. Các "đuôi" luôn **giảm dần** ⇒ tìm bằng nhị phân.
4. Số dãy mở = độ dài dãy con không tăng dài nhất (Mirsky/Dilworth).
5. Kiểm chứng bằng DP bitmask phân hoạch tối ưu.

## 2. Đề bài & ví dụ

Cho `nums`. Mỗi thao tác: xóa một **dãy con tăng nghiêm ngặt** khỏi mảng. Tìm số thao tác ít nhất để mảng rỗng.

**Ví dụ 1:** `[5,3,1,4,2]` → `3` (`[1,2]`, `[3,4]`, `[5]`). **Ví dụ 2:** `[1,2,3,4,5]` → `1`. **Ví dụ 3:** `[5,4,3,2,1]` → `5`.

**Ràng buộc (theo doocs):** `1 ≤ n ≤ 10^5`; `1 ≤ nums[i] ≤ 10^5`.

In [1]:
from typing import List
import bisect

examples = [([5, 3, 1, 4, 2], 3), ([1, 2, 3, 4, 5], 1), ([5, 4, 3, 2, 1], 5)]
for nums, exp in examples:
    print(nums, '-> kỳ vọng', exp)

[5, 3, 1, 4, 2] -> kỳ vọng 3
[1, 2, 3, 4, 5] -> kỳ vọng 1
[5, 4, 3, 2, 1] -> kỳ vọng 5


## 3. Kiến thức nền

### 3.1. Cận dưới từ dãy con không tăng

Nếu `i₁ < i₂ < … < i_L` và `nums[i₁] ≥ nums[i₂] ≥ … ≥ nums[i_L]` thì không hai phần tử nào trong đó cùng một dãy tăng nghiêm ngặt ⇒ cần `≥ L` dãy.

### 3.2. Dãy con không tăng dài nhất bằng patience

Đổi dấu: dãy không tăng của `nums` ⇔ dãy **không giảm** của `−nums` ⇒ LIS không nghiêm ngặt (`bisect_right`).

In [2]:
def longest_non_increasing(a):
    tails = []
    for x in a:
        v = -x
        i = bisect.bisect_right(tails, v)
        if i == len(tails):
            tails.append(v)
        else:
            tails[i] = v
    return len(tails)

print([longest_non_increasing(a) for a, _ in examples], longest_non_increasing([2, 2, 2]))

[3, 1, 5] 3


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Ví dụ 1 `[5, 3, 1, 4, 2]`, duy trì các đuôi (giảm dần):

| `x` | đuôi trước | đặt vào | đuôi sau |
|---|---|---|---|
| 5 | — | dãy mới | `[5]` |
| 3 | `[5]` | không đuôi nào `< 3` ⇒ mới | `[5, 3]` |
| 1 | `[5, 3]` | mới | `[5, 3, 1]` |
| 4 | `[5, 3, 1]` | đuôi lớn nhất `< 4` là `3` | `[5, 4, 1]` |
| 2 | `[5, 4, 1]` | đuôi `1` | `[5, 4, 2]` |

3 dãy ✓ — và `5, 3, 1` (hoặc `5, 4, 2`) là dãy không tăng dài 3.

## 5. Cách 1 — Brute Force: DP bitmask phân hoạch tối ưu

In [3]:
from functools import lru_cache
def brute(nums):
    n = len(nums)
    full = (1 << n) - 1
    def increasing(mask):
        prev = None
        for i in range(n):
            if mask >> i & 1:
                if prev is not None and nums[i] <= prev:
                    return False
                prev = nums[i]
        return True
    good = [increasing(m) for m in range(1 << n)]

    @lru_cache(maxsize=None)
    def f(rest):
        if rest == 0:
            return 0
        low = rest & -rest                     # phần tử đầu tiên còn lại phải thuộc một dãy nào đó
        best = n
        sub = rest
        while sub:
            if sub & low and good[sub]:
                best = min(best, 1 + f(rest ^ sub))
            sub = (sub - 1) & rest
        return best

    return f(full)

for nums, exp in examples:
    print(brute(nums), '| kỳ vọng', exp)

3 | kỳ vọng 3
1 | kỳ vọng 1
5 | kỳ vọng 5


**Độ phức tạp:** `O(3^n)` — chỉ `n ≤ 12`. Cố định phần tử thấp nhất còn lại thuộc dãy đang chọn để tránh đếm lặp.

## 6. Ý tưởng thuật toán / bước đột phá

```
tails = []                 # giảm dần
for x in nums:
    i = vị trí đầu tiên có tails[i] < x
    nếu i == len(tails): tails.append(x)  (mở dãy mới)
    ngược lại: tails[i] = x               (nối x vào dãy đó)
return len(tails)
```

Vì `tails` giảm dần, "đuôi lớn nhất `< x`" chính là phần tử **đầu tiên** `< x`.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — DP `O(n²)` dãy con không tăng dài nhất

In [4]:
def dp_quadratic(nums):
    n = len(nums); d = [1] * n
    for i in range(n):
        for j in range(i):
            if nums[j] >= nums[i] and d[j] + 1 > d[i]:
                d[i] = d[j] + 1
    return max(d)

for nums, exp in examples:
    print(dp_quadratic(nums), '| kỳ vọng', exp)

3 | kỳ vọng 3
1 | kỳ vọng 1
5 | kỳ vọng 5


### 7.2. Cách 3 — Tham lam xếp dãy + nhị phân (lời giải chính)

In [5]:
def greedy_chains(nums):
    tails = []
    for x in nums:
        lo, hi = 0, len(tails)
        while lo < hi:
            mid = (lo + hi) // 2
            if tails[mid] < x:
                hi = mid
            else:
                lo = mid + 1
        if lo == len(tails):
            tails.append(x)
        else:
            tails[lo] = x
    return len(tails)

for nums, exp in examples:
    print(greedy_chains(nums), '| kỳ vọng', exp)

3 | kỳ vọng 3
1 | kỳ vọng 1
5 | kỳ vọng 5


### 7.3. Cách 4 — Patience trên `−nums` (dãy không tăng dài nhất)

Dùng `longest_non_increasing` ở mục 3.2 — cùng độ phức tạp, minh họa trực tiếp đẳng thức Mirsky.

In [6]:
for nums, exp in examples:
    print(longest_non_increasing(nums), '| kỳ vọng', exp)

3 | kỳ vọng 3
1 | kỳ vọng 1
5 | kỳ vọng 5


## 8. TRACE từng bước

Trace cách 3 trên `[3, 1, 2, 2, 4, 1, 5]`, in cả các dãy được dựng.

In [7]:
def trace(nums):
    tails, chains = [], []
    for x in nums:
        i = next((k for k, t in enumerate(tails) if t < x), len(tails))
        if i == len(tails):
            tails.append(x); chains.append([x]); act = 'mở dãy mới'
        else:
            tails[i] = x; chains[i].append(x); act = f'nối vào dãy {i}'
        print(f'x = {x}: {act:12s} | đuôi {tails}')
    print('các dãy:', chains, '| số dãy:', len(chains), '| vét cạn:', brute(nums))

trace([3, 1, 2, 2, 4, 1, 5])

x = 3: mở dãy mới   | đuôi [3]
x = 1: mở dãy mới   | đuôi [3, 1]
x = 2: nối vào dãy 1 | đuôi [3, 2]
x = 2: mở dãy mới   | đuôi [3, 2, 2]
x = 4: nối vào dãy 0 | đuôi [4, 2, 2]
x = 1: mở dãy mới   | đuôi [4, 2, 2, 1]
x = 5: nối vào dãy 0 | đuôi [5, 2, 2, 1]
các dãy: [[3, 4, 5], [1, 2], [2], [1]] | số dãy: 4 | vét cạn: 4


## 9. Code nộp LeetCode

(Chữ ký theo doocs — xem ghi chú Premium.)

In [8]:
class Solution:
    def minOperations(self, nums: List[int]) -> int:
        neg_tails = []                         # lưu -đuôi: tăng dần, dùng bisect
        for x in nums:
            i = bisect.bisect_right(neg_tails, -x)  # đuôi đầu tiên < x  <=>  -đuôi > -x
            if i == len(neg_tails):
                neg_tails.append(-x)
            else:
                neg_tails[i] = -x
        return len(neg_tails)

for nums, exp in examples:
    got = Solution().minOperations(nums)
    print(got, 'OK' if got == exp else 'SAI')

3 OK
1 OK
5 OK


Lưu `−đuôi` để mảng **tăng dần**, dùng được `bisect` chuẩn: "đuôi `t < x`" ⇔ `−t > −x` ⇒ vị trí đầu tiên là `bisect_right(neg, −x)`.

## 10. Vì sao thuật toán đúng

1. **Hợp lệ:** mỗi lần nối `x` vào dãy có đuôi `t < x` giữ dãy tăng nghiêm ngặt.
2. **Bất biến:** đuôi giảm dần — thay `tails[i]` (phần tử đầu tiên `< x`) bằng `x`: `tails[i−1] ≥ x` (vì `tails[i−1]` không `< x`) và `x > tails[i] > tails[i+1]` ⇒ vẫn giảm (không nghiêm ngặt ở chỗ `tails[i−1] = x` có thể xảy ra).
3. **Tối ưu:** gắn cho mỗi phần tử `x` khi được đặt vào dãy số `i` (0-based) một "con trỏ" tới đuôi hiện tại của dãy `i − 1` (phần tử đứng trước `x` và `≥ x`, vì nó không `< x`). Lần theo con trỏ từ phần tử cuối của dãy cuối cùng thu được một dãy con **không tăng** có độ dài bằng số dãy. Kết hợp cận dưới (mục 3.1) ⇒ số dãy là tối ưu và bằng độ dài dãy không tăng dài nhất.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. DP bitmask | `O(3^n)` | `O(2^n)` |
| 2. DP `O(n²)` | `O(n²)` | `O(n)` |
| 3. Tham lam + nhị phân | `O(n log n)` | `O(n)` |
| 4. Patience trên `−nums` | `O(n log n)` | `O(n)` |

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên

In [9]:
impls = {'dp_quadratic': dp_quadratic, 'greedy_chains': greedy_chains, 'patience_neg': longest_non_increasing,
         'submit': lambda a: Solution().minOperations(a)}
edge = [([7], 1), ([2, 2], 2), ([1, 1, 2, 2], 2), ([3, 1, 2], 2), ([2, 1, 2, 1], 3)]
for nums, exp in examples + edge:
    assert brute(nums) == exp, (nums, brute(nums))
    for name, f in impls.items():
        assert f(nums) == exp, (name, nums)
print('Ví dụ đề và ca biên: tất cả đúng.')

Ví dụ đề và ca biên: tất cả đúng.


### 12.2. Đối chiếu ngẫu nhiên với DP bitmask

In [10]:
import random
rng = random.Random(3231)
cnt = 0
for _ in range(300):
    nums = [rng.randint(1, rng.choice([3, 6, 20])) for _ in range(rng.randint(1, 10))]
    exp = brute(nums)
    for name, f in impls.items():
        assert f(nums) == exp, (name, nums)
    cnt += 1
print('Khớp DP bitmask trên', cnt, 'mảng ngẫu nhiên (nhiều giá trị trùng).')

Khớp DP bitmask trên 300 mảng ngẫu nhiên (nhiều giá trị trùng).


### 12.3. Ca lớn `n = 10^5`

In [11]:
import time
N = 10 ** 5
big = {
    'ngẫu nhiên': [rng.randint(1, 10 ** 5) for _ in range(N)],
    'giảm dần': list(range(N, 0, -1)),
    'toàn bằng nhau': [7] * N,
    'tăng dần': list(range(1, N + 1)),
}
for name, nums in big.items():
    t = time.time(); a = Solution().minOperations(nums); t1 = time.time() - t
    b = greedy_chains(nums)
    assert a == b
    print(f"{name:14s}: {a:6d} | nộp {t1:.2f}s")
assert Solution().minOperations([7] * N) == N and Solution().minOperations(list(range(1, N + 1))) == 1
print('toàn bằng nhau -> N; tăng dần -> 1: đúng.')

ngẫu nhiên    :    622 | nộp 0.03s


giảm dần      : 100000 | nộp 0.03s


toàn bằng nhau: 100000 | nộp 0.02s
tăng dần      :      1 | nộp 0.01s
toàn bằng nhau -> N; tăng dần -> 1: đúng.


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Đếm dãy con **giảm nghiêm ngặt** dài nhất (quên phần tử bằng nhau)

In [12]:
def wrong_strict_decreasing(nums):
    tails = []
    for x in nums:
        i = bisect.bisect_left(tails, -x)                     # SAI: chỉ giảm nghiêm ngặt
        if i == len(tails):
            tails.append(-x)
        else:
            tails[i] = -x
    return len(tails)

print('sai :', wrong_strict_decreasing([2, 2]), '| đúng:', Solution().minOperations([2, 2]))

sai : 1 | đúng: 2


**Vì sao sai:** hai số bằng nhau cũng không thể cùng một dãy **tăng nghiêm ngặt** ⇒ cần đếm dãy **không tăng**.

### Lỗi 2 — Cho phép nối khi `đuôi ≤ x` (dãy chỉ không giảm)

In [13]:
def wrong_non_strict_chain(nums):
    tails = []
    for x in nums:
        i = next((k for k, t in enumerate(tails) if t <= x), len(tails))   # SAI
        if i == len(tails):
            tails.append(x)
        else:
            tails[i] = x
    return len(tails)

print('sai :', wrong_non_strict_chain([1, 1, 2, 2]), '| đúng:', Solution().minOperations([1, 1, 2, 2]))

sai : 1 | đúng: 2


**Vì sao sai:** bản sai cho phép dãy `1, 1, 2, 2` (không tăng nghiêm ngặt).

### Lỗi 3 — Nối vào dãy có đuôi **nhỏ nhất** `< x` (khớp tệ nhất)

In [14]:
def wrong_worst_fit(nums):
    tails = []
    for x in nums:
        cand = [k for k, t in enumerate(tails) if t < x]
        if not cand:
            tails.append(x)
        else:
            k = min(cand, key=lambda t: tails[t])                    # SAI: phung phí đuôi nhỏ
            tails[k] = x
    return len(tails)

found = None
r_ = random.Random(5)
for _ in range(5000):
    a = [r_.randint(1, 6) for _ in range(r_.randint(2, 7))]
    if wrong_worst_fit(a) != brute(a):
        found = a; break
print('phản ví dụ:', found, '-> sai:', wrong_worst_fit(found), '| đúng:', Solution().minOperations(found))

phản ví dụ: [1, 4, 1, 6, 2, 5] -> sai: 3 | đúng: 2


**Vì sao sai:** dùng đuôi nhỏ cho một `x` lớn làm "lãng phí" chỗ cho các phần tử nhỏ đến sau — phải chọn đuôi **lớn nhất** còn `< x`.

### Lỗi 4 — Đếm số lần giảm (`nums[i] ≤ nums[i−1]`) + 1

In [15]:
def wrong_count_descents(nums):
    return 1 + sum(1 for a, b in zip(nums, nums[1:]) if b <= a)        # SAI

print('sai :', wrong_count_descents([2, 3, 1, 4, 2]), '| đúng:', Solution().minOperations([2, 3, 1, 4, 2]))

sai : 3 | đúng: 2


**Vì sao sai:** đó là số **đoạn con liên tiếp** tăng; dãy con được bỏ qua phần tử, nên có thể ghép các đoạn.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Phủ/phân hoạch bằng ít dãy con tăng nhất" ⇒ Dilworth: bằng độ dài **antichain** dài nhất = dãy con theo chiều ngược lại.
- Tham lam "xếp bài" với đuôi đơn điệu ⇒ nhị phân.
- Cẩn thận nghiêm ngặt / không nghiêm ngặt ⇒ `bisect_left` vs `bisect_right`.

## 15. Biến thể của bài

1. Xóa dãy con **không giảm** ⇒ đáp án = dãy con **giảm nghiêm ngặt** dài nhất.
2. Xóa dãy con **giảm** nghiêm ngặt ⇒ dãy con không giảm dài nhất.
3. Dựng luôn các dãy (đã làm trong `trace`).
4. Mảng 2D (cặp `(a, b)`), xóa chuỗi tăng theo cả hai ⇒ Dilworth trên poset 2 chiều.

In [16]:
# Biến thể 1 và 2
def min_non_decreasing_cover(nums):
    tails = []
    for x in nums:
        i = bisect.bisect_left(tails, -x)       # dãy giảm nghiêm ngặt dài nhất
        if i == len(tails):
            tails.append(-x)
        else:
            tails[i] = -x
    return len(tails)

def min_decreasing_cover(nums):
    tails = []
    for x in nums:
        i = bisect.bisect_right(tails, x)       # dãy không giảm dài nhất
        if i == len(tails):
            tails.append(x)
        else:
            tails[i] = x
    return len(tails)

def cover_brute(nums, ok):
    n = len(nums)
    good = [all(ok(nums[i], nums[j]) for i in range(n) for j in range(i + 1, n) if m >> i & 1 and m >> j & 1) for m in range(1 << n)]
    @lru_cache(maxsize=None)
    def f(rest):
        if not rest:
            return 0
        low = rest & -rest; best = n; sub = rest
        while sub:
            if sub & low and good[sub]:
                best = min(best, 1 + f(rest ^ sub))
            sub = (sub - 1) & rest
        return best
    return f((1 << n) - 1)

rng = random.Random(15)
for _ in range(150):
    a = [rng.randint(1, 5) for _ in range(rng.randint(1, 8))]
    assert min_non_decreasing_cover(a) == cover_brute(a, lambda u, v: u <= v)
    assert min_decreasing_cover(a) == cover_brute(a, lambda u, v: u > v)
print('Biến thể 1, 2 khớp vét cạn.')

Biến thể 1, 2 khớp vét cạn.


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Ghi chú |
|---|---|---|---|
| 1 | DP bitmask phân hoạch | `O(3^n)` | Oracle |
| 2 | DP `O(n²)` không tăng dài nhất | `O(n²)` | Mirsky |
| 3 | Tham lam xếp dãy | `O(n log n)` | **Nộp bài** |
| 4 | Patience trên `−nums` | `O(n log n)` | Gọn |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `[1, 3, 2]`.

**Bài 2 (tính tay):** `[4, 4, 1, 1]`.

**Bài 3 (điền TODO):** trả về luôn cách phân hoạch tối ưu (danh sách các dãy theo chỉ số).

**Bài 4 (tự viết lại):** biến thể 4 — phần tử là cặp `(a, b)`, dãy "tăng" nếu cả hai thành phần tăng nghiêm ngặt và chỉ số tăng.

**Bài 5 (mini challenge):** số thao tác ít nhất nếu mỗi thao tác được xóa **tối đa `k`** phần tử tăng nghiêm ngặt.

In [17]:
def partition_todo(nums):
    # TODO
    raise NotImplementedError

def pairs_cover_todo(pairs):
    # TODO
    raise NotImplementedError

def bounded_k_todo(nums, k):
    # TODO
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Dãy không tăng dài nhất: `3, 2`. (2) Hai dãy: `[1, 3]`, `[2]`. (3) `2`.

**Bài 2.** (1) `4, 4, 1, 1` là dãy không tăng. (2) Bốn phần tử. (3) `4`.

**Bài 3.** (1) Như `trace`, lưu chỉ số. (2) Mỗi dãy là danh sách chỉ số. (3) Kiểm mỗi dãy tăng nghiêm ngặt.

**Bài 4.** (1) Antichain trong poset 3 chiều (chỉ số, a, b). (2) Không còn tham lam đơn giản — dùng DP bitmask cho `n` nhỏ hoặc ghép cặp (Dilworth ⇒ `n − ghép cực đại` trên đồ thị so sánh được). (3) Vét cạn đối chiếu.

**Bài 5.** (1) Mỗi thao tác tối đa `k` phần tử ⇒ cần `≥ ⌈n/k⌉`. (2) Cần `≥` đáp án gốc `L`. (3) Đoán `max(⌈n/k⌉, L)` là đủ? Vét cạn cho thấy **không** — bài toán phủ chuỗi có giới hạn độ dài khó hơn nhiều; dùng DP bitmask cho `n` nhỏ.

## 19. Đáp án tham khảo cho bài tập

In [18]:
# Bài 1, 2, 3
assert Solution().minOperations([1, 3, 2]) == 2
assert Solution().minOperations([4, 4, 1, 1]) == 4

def partition(nums):
    tails, chains = [], []
    for idx, x in enumerate(nums):
        i = next((k for k, t in enumerate(tails) if t < x), len(tails))
        if i == len(tails):
            tails.append(x); chains.append([idx])
        else:
            tails[i] = x; chains[i].append(idx)
    return chains

rng = random.Random(3)
for _ in range(200):
    a = [rng.randint(1, 6) for _ in range(rng.randint(1, 9))]
    ch = partition(a)
    assert len(ch) == Solution().minOperations(a)
    assert sorted(i for c in ch for i in c) == list(range(len(a)))
    assert all(a[c[t]] < a[c[t + 1]] for c in ch for t in range(len(c) - 1))
print('Bài 1: 2 | Bài 2: 4 | Bài 3: phân hoạch hợp lệ và tối ưu. Ví dụ:', partition(examples[0][0]))

Bài 1: 2 | Bài 2: 4 | Bài 3: phân hoạch hợp lệ và tối ưu. Ví dụ: [[0], [1, 3], [2, 4]]


In [19]:
# Bài 4, 5
def pairs_cover(pairs):
    n = len(pairs)
    ok = lambda i, j: pairs[i][0] < pairs[j][0] and pairs[i][1] < pairs[j][1]     # i < j
    # Dilworth: số chuỗi ít nhất = n − ghép cực đại trên đồ thị (i -> j nếu ok)
    match = [-1] * n
    def try_k(u, seen):
        for v in range(u + 1, n):
            if ok(u, v) and not seen[v]:
                seen[v] = True
                if match[v] == -1 or try_k(match[v], seen):
                    match[v] = u
                    return True
        return False
    m = sum(1 for u in range(n) if try_k(u, [False] * n))
    return n - m

def bounded_k(nums, k):
    n = len(nums)
    good = [all(nums[i] < nums[j] for i in range(n) for j in range(i + 1, n) if mm >> i & 1 and mm >> j & 1) and bin(mm).count('1') <= k
            for mm in range(1 << n)]
    @lru_cache(maxsize=None)
    def f(rest):
        if not rest:
            return 0
        low = rest & -rest; best = n; sub = rest
        while sub:
            if sub & low and good[sub]:
                best = min(best, 1 + f(rest ^ sub))
            sub = (sub - 1) & rest
        return best
    return f((1 << n) - 1)

rng = random.Random(4)
for _ in range(100):
    ps = [(rng.randint(1, 5), rng.randint(1, 5)) for _ in range(rng.randint(1, 7))]
    n = len(ps)
    good = [all(ps[i][0] < ps[j][0] and ps[i][1] < ps[j][1] for i in range(n) for j in range(i + 1, n) if m >> i & 1 and m >> j & 1) for m in range(1 << n)]
    @lru_cache(maxsize=None)
    def g(rest):
        if not rest:
            return 0
        low = rest & -rest; best = n; sub = rest
        while sub:
            if sub & low and good[sub]:
                best = min(best, 1 + g(rest ^ sub))
            sub = (sub - 1) & rest
        return best
    assert pairs_cover(ps) == g((1 << n) - 1)
    a = [rng.randint(1, 5) for _ in range(rng.randint(1, 8))]; k = rng.randint(1, 4)
    assert bounded_k(a, k) >= max(-(-len(a) // k), Solution().minOperations(a))
gap = None
for _ in range(3000):
    a = [rng.randint(1, 4) for _ in range(rng.randint(3, 8))]; k = rng.randint(2, 3)
    if bounded_k(a, k) > max(-(-len(a) // k), Solution().minOperations(a)):
        gap = (a, k); break
print('Bài 4 (Dilworth bằng ghép cặp) khớp vét cạn.')
print('Bài 5: cận dưới max(ceil(n/k), L) luôn đúng nhưng KHÔNG luôn đạt — ví dụ', gap,
      '-> đáp án', bounded_k(*gap), '> cận', max(-(-len(gap[0]) // gap[1]), Solution().minOperations(gap[0])))

Bài 4 (Dilworth bằng ghép cặp) khớp vét cạn.
Bài 5: cận dưới max(ceil(n/k), L) luôn đúng nhưng KHÔNG luôn đạt — ví dụ ([1, 2, 2, 3, 4, 1], 2) -> đáp án 4 > cận 3


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
neg = []
for x in nums:
    i = bisect_right(neg, -x)        # đuôi đầu tiên < x
    neg.append(-x) if i == len(neg) else neg.__setitem__(i, -x)
return len(neg)                      # = dãy con không tăng dài nhất
```

**Checklist:**

- [ ] Dãy xóa tăng **nghiêm ngặt** ⇒ đếm dãy **không tăng** (kể cả bằng nhau)?
- [ ] Nối vào đuôi **lớn nhất** còn `< x`?
- [ ] `bisect_right` trên `−đuôi`?
- [ ] Không nhầm với đếm đoạn tăng liên tiếp?
- [ ] Mảng toàn bằng nhau ⇒ `n`?